# Genesis — IE 643 | Day 5: final-method showcase (C_anchor + LoRA) + [CUT] demo + metrics on Day 2/3

**Before you run — this is what went wrong on Day 4.** No input was attached, so the notebook found neither the
Day 2/3 videos nor the LoRA. This notebook now **stops within ~2 minutes** if they are missing, so no GPU time is wasted.
1. Right panel → **Input → + Add Input → Your Work → Notebooks** → add **Day 2**, **Day 3** (and optionally **Day 4**).
   After adding, the Input panel must list them; step 3 prints what it found.
2. Settings: Accelerator **GPU T4 x2** · Internet **On** → **Save Version → Save & Run All (Commit)**. About 2.5–3 h.

**Changes from Day 4**
- New fox prompts: **side view, walking left to right**, gentle actions. In Day 4 the first clip showed the fox from
  behind (small, tail towards the camera); the model later re-read the tail as a head and the fox "flipped".
  Big actions (pounce, dig, shake) also made clips 3–5 unstable (highest motion, lowest quality score).
- Priority: final method first, then the [CUT] demo, then the no-LoRA ablation from the same first clip.
- `hf_xet` removed and leftover processes killed at the end (suspected cause of the long "still running" after finishing).

## 1 — Install

In [ ]:
!pip uninstall -y -q torchao
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg huggingface_hub peft pandas matplotlib psutil
!pip uninstall -y -q hf_xet
import os; os.environ["HF_HUB_DISABLE_XET"] = "1"
import diffusers, transformers, torch, peft
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| torch", torch.__version__, "| GPUs", torch.cuda.device_count())

## 2 — Project code (saved into the output; part of the submission)

In [ ]:
import os; os.makedirs("/kaggle/working/genesis5/code", exist_ok=True)

In [ ]:
%%writefile /kaggle/working/genesis5/code/genesis_lib.py
"""
genesis_lib.py — Genesis (IE 643): long-form video by chaining short Wan2.2 TI2V-5B clips.

Chaining methods
  baseline : every clip generated independently from text (T2V) and joined with hard cuts.
  A (m=1)  : clip k+1 is conditioned on the LAST FRAME of clip k (the model's own TI2V first-frame conditioning).
  B (m>=2) : clip k+1 is conditioned on the last K = 1 + 4(m-1) FRAMES of clip k. They are re-encoded by the
             causal 3D VAE into m clean latent frames, placed at the start of the new clip's latent; those tokens
             get flow-matching timestep 0 (clean) and are re-injected after every solver step, so the model
             denoises the rest of the clip around a context that shows the motion, not just a still frame.
  C        : Method B + drift control. (i) every generated frame is colour-anchored to clip 0 (its LAB mean and
             standard deviation are pulled toward clip 0's by a factor `anchor`), so exposure and saturation
             cannot creep upward clip after clip; (ii) continuation clips use a lower CFG scale (`cfg_cont`),
             because guidance applied on already-saturated context compounds the drift.
  +LoRA    : any chained method with the continuation LoRA (genesis_train_lora.py) switched on.

Flow matching (what the model was trained with)
  x_s = (1 - s) * x_0 + s * eps,  s in [0, 1]   (s = 1 pure noise, s = 0 clean)
  model predicts velocity v = eps - x_0;  loss = || (eps - x_0) - v_theta(x_s, s, text) ||^2
  TI2V-5B gives every latent token its own timestep: conditioning tokens get s = 0, the rest the current s.
"""
import json, os, time

import numpy as np
import torch
import diffusers
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline, WanPipeline, WanTransformer3DModel
from diffusers.utils import export_to_video
from diffusers.utils.torch_utils import get_module_execution_device, randn_tensor
from diffusers.pipelines.wan.pipeline_wan_i2v import retrieve_latents
from PIL import Image

TEMPORAL = 4      # Wan2.2 VAE temporal compression
TAIL_KEEP = 9     # pixel frames kept losslessly per clip = context for m <= 3
SAVE_LATENTS = True   # Day 4+: set False to keep the notebook output small


def context_frames(m: int) -> int:
    """Pixel frames the VAE maps to m latent frames (first frame alone, then groups of 4)."""
    return 1 + TEMPORAL * (m - 1)


def new_frames_per_clip(num_frames: int, m: int) -> int:
    """Frames a chained clip adds to the video (its first K frames repeat the context and are dropped)."""
    return num_frames - context_frames(m)


def clips_for_duration(seconds: float, fps: int, num_frames: int, m: int) -> int:
    """Number of clips needed so the stitched video lasts at least `seconds`."""
    need = max(int(round(seconds * fps)), num_frames)
    return 1 + int(np.ceil((need - num_frames) / new_frames_per_clip(num_frames, m)))


def _prepare_latents_ctx(self, image, batch_size, num_channels_latents=48, height=480, width=832,
                         num_frames=49, dtype=None, device=None, generator=None, latents=None, last_image=None):
    """Replacement for WanImageToVideoPipeline.prepare_latents (expand_timesteps / TI2V path).

    If self._genesis_ctx (uint8 array, K x H x W x 3) is set, it is encoded into m = (K-1)/4 + 1 clean latent
    frames and the first m latent frames are masked as 'known' (timestep 0, re-injected every step).
    Otherwise falls back to the stock single-image behaviour.
    """
    ctx = getattr(self, "_genesis_ctx", None)
    if ctx is None:
        return WanImageToVideoPipeline.prepare_latents(self, image, batch_size, num_channels_latents, height, width,
                                                       num_frames, dtype, device, generator, latents, last_image)
    K = ctx.shape[0]
    assert (K - 1) % TEMPORAL == 0, f"context length {K} must be 1 + 4(m-1)"
    assert ctx.shape[1:3] == (height, width), f"context frames {ctx.shape[1:3]} != ({height}, {width})"

    T = (num_frames - 1) // self.vae_scale_factor_temporal + 1
    h, w = height // self.vae_scale_factor_spatial, width // self.vae_scale_factor_spatial
    shape = (batch_size, num_channels_latents, T, h, w)
    if latents is None:
        latents = randn_tensor(shape, generator=generator, device=device, dtype=dtype)
    else:
        latents = latents.to(device=device, dtype=dtype)

    vid = torch.from_numpy(np.ascontiguousarray(ctx)).float().div(127.5).sub(1.0)   # K,H,W,3 in [-1, 1]
    vid = vid.permute(3, 0, 1, 2).unsqueeze(0)                                     # 1,3,K,H,W
    vae_dev = get_module_execution_device(self.vae)
    with torch.no_grad():
        lc = retrieve_latents(self.vae.encode(vid.to(vae_dev, self.vae.dtype)), sample_mode="argmax")
    lc = lc.to(device=device, dtype=dtype)
    mean = torch.tensor(self.vae.config.latents_mean).view(1, -1, 1, 1, 1).to(device, dtype)
    inv_std = 1.0 / torch.tensor(self.vae.config.latents_std).view(1, -1, 1, 1, 1).to(device, dtype)
    lc = (lc - mean) * inv_std

    m = lc.shape[2]
    assert m < T, "context must be shorter than the clip"
    cond = torch.zeros_like(latents)
    cond[:, :, :m] = lc.expand(batch_size, -1, -1, -1, -1)
    mask = torch.ones(1, 1, T, h, w, dtype=dtype, device=device)
    mask[:, :, :m] = 0
    if getattr(self, "_genesis_vae_offload", False):
        self.vae.to("cpu")
        torch.cuda.empty_cache()
    return latents, cond, mask


class _T2V(WanPipeline):
    """Stock T2V pipeline, but always computes on the GPU even while the VAE sits on the CPU."""
    @property
    def _execution_device(self):
        return torch.device(self._genesis_device)


class _CtxI2V(WanImageToVideoPipeline):
    """TI2V pipeline whose conditioning can be a multi-frame context (see _prepare_latents_ctx)."""
    _genesis_ctx = None
    _genesis_vae_offload = False
    prepare_latents = _prepare_latents_ctx

    @property
    def _execution_device(self):
        return torch.device(self._genesis_device)


class Genesis:
    """Holds one TI2V-5B transformer + VAE on one GPU and generates / chains clips."""

    def __init__(self, weights_dir, height=480, width=832, num_frames=49, steps=30, guidance=5.0,
                 flow_shift=3.0, fps=24, device="cuda", vae_offload=True, dtype=torch.float16):
        self.h, self.w, self.nf, self.steps, self.cfg, self.fps = height, width, num_frames, steps, guidance, fps
        self.device, self.vae_offload, self.dtype = device, vae_offload, dtype
        idx = json.load(open(f"{weights_dir}/model_index.json"))
        expand = bool(idx.get("expand_timesteps", False))
        assert expand, "TI2V-5B must load with expand_timesteps=True (per-token timesteps)"

        self.vae = AutoencoderKLWan.from_pretrained(f"{weights_dir}/vae", torch_dtype=torch.float32)
        self.vae.enable_tiling()
        self.vae.to("cpu" if vae_offload else device)
        try:
            self.transformer = WanTransformer3DModel.from_pretrained(
                f"{weights_dir}/transformer", torch_dtype=dtype, device_map=device)
        except Exception as e:  # older/newer API without device_map support for this model
            print("device_map load failed, loading on CPU first:", repr(e)[:200])
            self.transformer = WanTransformer3DModel.from_pretrained(
                f"{weights_dir}/transformer", torch_dtype=dtype).to(device)
        self.transformer.eval()

        cfg = json.load(open(f"{weights_dir}/scheduler/scheduler_config.json"))
        cls = getattr(diffusers, cfg["_class_name"])
        sched = cls.from_pretrained(f"{weights_dir}/scheduler")
        key = "flow_shift" if "flow_shift" in sched.config else ("shift" if "shift" in sched.config else None)
        if key:
            sched = cls.from_config(sched.config, **{key: flow_shift})

        common = dict(tokenizer=None, text_encoder=None, vae=self.vae, scheduler=sched,
                      transformer=self.transformer, expand_timesteps=expand)
        self.t2v = _T2V(**common)
        self.i2v = _CtxI2V(**common)
        self.i2v._genesis_vae_offload = vae_offload
        for p in (self.t2v, self.i2v):
            p._genesis_device = device
            p.set_progress_bar_config(disable=True)
        self._lora_path = None      # adapter loaded into the transformer (None = never loaded)
        self.lora_active = False

    # ---------------------------------------------------------------- LoRA
    def set_lora(self, path):
        """path=None switches the adapter off; a directory with lora.safetensors + lora_config.json switches it on."""
        if path is None:
            if self._lora_path is not None and self.lora_active:
                self.transformer.disable_adapters()
            self.lora_active = False
            return
        if self._lora_path is None:
            from peft import LoraConfig, set_peft_model_state_dict
            from safetensors.torch import load_file
            cfg = json.load(open(f"{path}/lora_config.json"))
            self.transformer.add_adapter(LoraConfig(**cfg), adapter_name="genesis")
            sd = load_file(f"{path}/lora.safetensors")
            res = set_peft_model_state_dict(self.transformer, sd, adapter_name="genesis")
            missing = [k for k in getattr(res, "unexpected_keys", [])]
            assert not missing, f"LoRA keys not loaded: {missing[:5]}"
            for n, prm in self.transformer.named_parameters():   # cast only the adapter: some base modules
                if "lora_" in n:                                    # (norms, time embedder) must stay fp32
                    prm.data = prm.data.to(self.dtype)
            self._lora_path = path
        elif path != self._lora_path:
            raise ValueError("a different LoRA is already loaded in this process")
        self.transformer.enable_adapters()
        self.lora_active = True

    # ---------------------------------------------------------------- helpers
    def _emb(self, E, key):
        return E[key].to(self.device, self.dtype)

    def _kwargs(self, E, seed, guidance=None):
        return dict(negative_prompt_embeds=self._emb(E, "__negative__"), height=self.h, width=self.w,
                    num_frames=self.nf, num_inference_steps=self.steps,
                    guidance_scale=self.cfg if guidance is None else guidance,
                    generator=torch.Generator("cpu").manual_seed(int(seed)), output_type="latent")

    @torch.no_grad()
    def _decode(self, lat):
        v = self.vae
        v.to(self.device)
        try:
            z = lat.to(self.device, v.dtype)
            mean = torch.tensor(v.config.latents_mean).view(1, -1, 1, 1, 1).to(z.device, z.dtype)
            std = torch.tensor(v.config.latents_std).view(1, -1, 1, 1, 1).to(z.device, z.dtype)
            video = v.decode(z * std + mean, return_dict=False)[0]
            return self.t2v.video_processor.postprocess_video(video, output_type="np")[0]  # F,H,W,3 in [0,1]
        finally:
            if self.vae_offload:
                v.to("cpu")
            torch.cuda.empty_cache()

    def decode(self, lat):
        try:
            return self._decode(lat)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            self.transformer.to("cpu")
            torch.cuda.empty_cache()
            try:
                return self._decode(lat)
            finally:
                self.transformer.to(self.device)

    # ---------------------------------------------------------------- generation
    def text_clip(self, E, key, seed):
        """Independent clip from text only (baseline, and clip 0 of every chain)."""
        lat = self.t2v(prompt_embeds=self._emb(E, key), **self._kwargs(E, seed)).frames
        return self.decode(lat), lat

    def context_clip(self, ctx_uint8, E, key, seed, guidance=None):
        """Clip conditioned on K = 1 + 4(m-1) previous frames (m = 1 is Method A, m >= 2 is Method B)."""
        if self.vae_offload:
            self.vae.to(self.device)
        self.i2v._genesis_ctx = ctx_uint8
        try:
            dummy = Image.fromarray(ctx_uint8[-1])   # required argument; real conditioning is _genesis_ctx
            lat = self.i2v(image=dummy, prompt_embeds=self._emb(E, key), **self._kwargs(E, seed, guidance)).frames
        finally:
            self.i2v._genesis_ctx = None
            if self.vae_offload:
                self.vae.to("cpu")
        return self.decode(lat), lat


# -------------------------------------------------------------------- storage
def to_uint8(frames):
    return (np.clip(frames, 0.0, 1.0) * 255.0).round().astype(np.uint8)


def frames_ok(frames):
    return bool(np.isfinite(frames).all() and frames.max() > 0.02)


def save_clip(out, tag, frames, lat, fps):
    export_to_video(list(frames), f"{out}/videos/{tag}.mp4", fps=fps)
    if SAVE_LATENTS:
        torch.save(lat.detach().cpu().half(), f"{out}/latents/{tag}.pt")
    np.save(f"{out}/frames/{tag}_tail.npy", to_uint8(frames[-TAIL_KEEP:]))


def clip_done(out, tag):
    return os.path.exists(f"{out}/frames/{tag}_tail.npy") and os.path.exists(f"{out}/videos/{tag}.mp4")


def load_tail(out, tag):
    return np.load(f"{out}/frames/{tag}_tail.npy")


def lab_stats(frames_u8):
    """Mean and std of each LAB channel over a stack of uint8 RGB frames (OpenCV LAB, 0-255 scale)."""
    import cv2
    lab = np.stack([cv2.cvtColor(f, cv2.COLOR_RGB2LAB) for f in frames_u8]).reshape(-1, 3).astype(np.float32)
    return lab.mean(0), lab.std(0) + 1e-6


def colour_anchor(frames, ref, strength):
    """Pull each frame's LAB mean/std toward the reference stats by `strength` in [0, 1]. frames: float [0,1]."""
    import cv2
    ref_mu, ref_sd = ref
    out = np.empty_like(frames)
    for j, f in enumerate(to_uint8(frames)):
        lab = cv2.cvtColor(f, cv2.COLOR_RGB2LAB).astype(np.float32)
        mu, sd = lab.reshape(-1, 3).mean(0), lab.reshape(-1, 3).std(0) + 1e-6
        t_mu, t_sd = mu + strength * (ref_mu - mu), sd + strength * (ref_sd - sd)
        lab = (lab - mu) / sd * t_sd + t_mu
        rgb = cv2.cvtColor(np.clip(lab, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB)
        out[j] = rgb.astype(np.float32) / 255.0
    return out


def read_video(path):
    import imageio.v3 as iio
    return np.stack(list(iio.imiter(path))).astype(np.float32) / 255.0


def log_jsonl(path, rec):
    with open(path, "a") as f:
        f.write(json.dumps(rec) + "\n")


# -------------------------------------------------------------------- one sequence, one method
def run_sequence(G, E, out, seq, n_clips, seeds, method, m, log_path, cfg_cont=None, anchor=0.0, lora=None):
    """method: 'baseline' or a chain name; m: context latent frames (1 = Method A);
    cfg_cont: CFG scale for continuation clips (None = same as clip 0); anchor: colour-anchoring strength;
    lora: LoRA directory to switch on for the continuation clips (None = off)."""
    clips = []
    ref = None
    for i in range(n_clips):
        if method == "baseline" or i == 0:
            tag = f"{seq}_baseline_c{i}"
        else:
            tag = f"{seq}_{method}_c{i}"
        if clip_done(out, tag):
            frames = read_video(f"{out}/videos/{tag}.mp4")
            stats = {"resumed": True}
        else:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0 = time.time()
            if method == "baseline" or i == 0:
                G.set_lora(None)                     # clip 0 is always the shared, unmodified baseline clip
                frames, lat = G.text_clip(E, f"{seq}_{i}", seeds[i])
            else:
                G.set_lora(lora)
                K = context_frames(m)
                ctx = load_tail(out, prev_tag)[-K:]
                frames, lat = G.context_clip(ctx, E, f"{seq}_{i}", seeds[i], guidance=cfg_cont)
                if anchor > 0:
                    if ref is None:
                        ref = lab_stats(to_uint8(read_video(f"{out}/videos/{seq}_baseline_c0.mp4")))
                    frames = np.concatenate([frames[:K], colour_anchor(frames[K:], ref, anchor)], axis=0)
            stats = {"sec": round(time.time() - t0, 1),
                     "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2) if torch.cuda.is_available() else 0.0,
                     "frames_ok": frames_ok(frames)}
            save_clip(out, tag, frames, lat, G.fps)
        drop = 0 if (method == "baseline" or i == 0) else context_frames(m)
        clips.append(frames[drop:])
        prev_tag = tag
        rec = {"seq": seq, "method": method, "m": m, "clip": i, "tag": tag, "cfg_cont": cfg_cont,
               "anchor": anchor, "lora": bool(lora) and i > 0, **stats}
        log_jsonl(log_path, rec)
        print(rec, flush=True)

    full = np.concatenate(clips, axis=0)
    export_to_video(list(full), f"{out}/videos/{seq}_{method}_FULL.mp4", fps=G.fps)
    lengths = [int(c.shape[0]) for c in clips]
    G.set_lora(None)
    json.dump({"seq": seq, "method": method, "m": m, "cfg_cont": cfg_cont, "anchor": anchor, "lora": bool(lora),
               "clip_lengths": lengths,
               "boundaries": [int(b) for b in np.cumsum(lengths)[:-1]], "seconds": round(full.shape[0] / G.fps, 2)},
              open(f"{out}/manifests/{seq}_{method}.json", "w"), indent=2)
    return full


# -------------------------------------------------------------------- a "script": several shots separated by [CUT]
def parse_script(prompts):
    """['p0', 'p1', '[CUT]', 'p2', ...] -> [{'prompt': 'p0', 'cut': False}, {'prompt': 'p1', 'cut': False},
    {'prompt': 'p2', 'cut': True}, ...]. A prompt may also start with '[CUT]' itself."""
    items, pending_cut = [], False
    for p in prompts:
        p = p.strip()
        while p.upper().startswith("[CUT]"):
            pending_cut, p = True, p[5:].strip()
        if not p:
            continue
        items.append({"prompt": p, "cut": bool(items) and pending_cut})
        pending_cut = False
    return items


def expand_script(items, n_clips):
    """Spread the script's prompts over n_clips clips: each shot keeps its prompts in order; when there are more
    clips than prompts, the clips are shared out as evenly as possible (each prompt repeats for a few clips)."""
    n = len(items)
    if n_clips <= n:
        return [dict(it) for it in items[:max(n_clips, 1)]]
    reps = [n_clips // n + (1 if i < n_clips % n else 0) for i in range(n)]
    out = []
    for it, r in zip(items, reps):
        for k in range(r):
            out.append(dict(it, cut=it["cut"] and k == 0))
    return out


def run_script(G, E, out, name, items, method, m, log_path, cfg_cont=None, anchor=0.0, lora=None, finalize=True):
    """Generalised run_sequence. items: [{'key': embedding key, 'seed': int, 'cut': bool, 'prompt': str}, ...].
    A clip with cut=True (and clip 0) starts a NEW SHOT: generated from text only (base model), giving a hard cut.
    Every other clip continues the previous one with the chosen method (m context latent frames, CFG for
    continuation clips, colour anchoring to the first clip of the CURRENT shot, optional LoRA).
    Shot-start clips are tagged {name}_text_c{i} so that different methods share them."""
    clips, cuts, ref, prev_tag, pos = [], [], None, None, 0
    for i, it in enumerate(items):
        new_shot = i == 0 or bool(it.get("cut"))
        tag = f"{name}_text_c{i}" if new_shot else f"{name}_{method}_c{i}"
        if new_shot:
            ref = None
            shot_first = tag
            if i > 0:
                cuts.append(pos)
        if clip_done(out, tag):
            frames = read_video(f"{out}/videos/{tag}.mp4")
            stats = {"resumed": True}
        else:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0 = time.time()
            if new_shot:
                G.set_lora(None)
                frames, lat = G.text_clip(E, it["key"], it["seed"])
            else:
                G.set_lora(lora)
                K = context_frames(m)
                frames, lat = G.context_clip(load_tail(out, prev_tag)[-K:], E, it["key"], it["seed"], guidance=cfg_cont)
                if anchor > 0:
                    if ref is None:
                        ref = lab_stats(to_uint8(read_video(f"{out}/videos/{shot_first}.mp4")))
                    frames = np.concatenate([frames[:K], colour_anchor(frames[K:], ref, anchor)], axis=0)
            stats = {"sec": round(time.time() - t0, 1),
                     "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2) if torch.cuda.is_available() else 0.0,
                     "frames_ok": frames_ok(frames)}
            save_clip(out, tag, frames, lat, G.fps)
        drop = 0 if new_shot else context_frames(m)
        clips.append(frames[drop:])
        pos += clips[-1].shape[0]
        prev_tag = tag
        rec = {"seq": name, "method": method, "m": m, "clip": i, "tag": tag, "cut": new_shot and i > 0,
               "cfg_cont": cfg_cont, "anchor": anchor, "lora": bool(lora) and not new_shot, **stats}
        log_jsonl(log_path, rec)
        print(rec, flush=True)
    G.set_lora(None)
    if not finalize:
        return None
    full = np.concatenate(clips, axis=0)
    export_to_video(list(full), f"{out}/videos/{name}_{method}_FULL.mp4", fps=G.fps)
    lengths = [int(c.shape[0]) for c in clips]
    json.dump({"seq": name, "method": method, "m": m, "cfg_cont": cfg_cont, "anchor": anchor, "lora": bool(lora),
               "clip_lengths": lengths, "boundaries": [int(b) for b in np.cumsum(lengths)[:-1]], "cuts": cuts,
               "prompts": [it.get("prompt", "") for it in items], "seconds": round(full.shape[0] / G.fps, 2)},
              open(f"{out}/manifests/{name}_{method}.json", "w"), indent=2)
    return full

In [ ]:
%%writefile /kaggle/working/genesis5/code/genesis_queue_worker.py
"""genesis_queue_worker.py (Day 4) — one process per GPU; pulls script jobs from a shared file queue.

Job = one script (list of clips, optionally split into shots by [CUT]) generated with one method.
A job runs once its dependencies are done. Jobs are claimed atomically with O_EXCL lock files.
Time budget: a job is only started if it can finish before config DEADLINE (epoch seconds), estimated as
(clips still to generate) x SEC_PER_CLIP; otherwise it is marked .skipped, so the commit never overruns.
"""
import argparse, json, os, sys, time, traceback
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import genesis_lib as gl

ap = argparse.ArgumentParser()
ap.add_argument("--plan", required=True)
ap.add_argument("--worker", required=True)
a = ap.parse_args()

plan = json.load(open(a.plan))
C = plan["config"]
OUT, Q = C["OUT"], f'{C["OUT"]}/queue'
os.makedirs(Q, exist_ok=True)
gl.SAVE_LATENTS = bool(C.get("SAVE_LATENTS", True))
DEADLINE, SEC_PER_CLIP = C.get("DEADLINE"), C.get("SEC_PER_CLIP", 480)


def st(name, kind):
    return os.path.exists(f"{Q}/{name}.{kind}")


def dep_state(d):
    if st(d, "done"):
        return "ok"
    if st(d, "failed") or st(d, "skipped"):
        return "dead"
    return "wait"


def clips_left(job):
    n = 0
    for i, it in enumerate(job["items"]):
        new_shot = i == 0 or it.get("cut")
        tag = f"{job['name']}_text_c{i}" if new_shot else f"{job['name']}_{job['method']}_c{i}"
        n += 0 if gl.clip_done(OUT, tag) else 1
    return n


def next_job():
    """Returns (job, None) to run, (None, 'wait') if something may become runnable, (None, 'exit') otherwise."""
    waiting = False
    for job in plan["jobs"]:
        j = job["id"]
        if any(st(j, k) for k in ("done", "failed", "skipped", "claimed")):
            continue
        deps = [dep_state(d) for d in job.get("deps", [])]
        if "dead" in deps:
            open(f"{Q}/{j}.skipped", "w").write("dependency failed or skipped")
            continue
        if "wait" in deps:
            waiting = True
            continue
        if DEADLINE and time.time() + clips_left(job) * SEC_PER_CLIP > DEADLINE:
            open(f"{Q}/{j}.skipped", "w").write("time budget")
            print(f"worker {a.worker}: skip {j} (would overrun the time budget)", flush=True)
            continue
        try:
            fd = os.open(f"{Q}/{j}.claimed", os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.write(fd, a.worker.encode()); os.close(fd)
            return job, None
        except FileExistsError:
            continue
    return None, ("wait" if waiting else "exit")


G = gl.Genesis(C["WEIGHTS_DIR"], C["HEIGHT"], C["WIDTH"], C["NUM_FRAMES"], C["STEPS"], C["GUIDANCE"],
               C["FLOW_SHIFT"], C["FPS"], device=C.get("DEVICE", "cuda"), vae_offload=C.get("VAE_OFFLOAD", True),
               dtype=getattr(torch, C.get("DTYPE", "float16")))
open(f"{OUT}/logs/worker{a.worker}.ready", "w").write("ok")
print(f"worker {a.worker} loaded", flush=True)
E = torch.load(C["EMBEDS"])
log = f"{OUT}/logs/runs_worker{a.worker}.jsonl"

while True:
    job, why = next_job()
    if job is None:
        if why == "exit":
            break
        time.sleep(60)
        continue
    print(f"worker {a.worker} -> {job['id']}", flush=True)
    try:
        lora = C["LORA_DIR"] if job.get("lora") else None
        gl.run_script(G, E, OUT, job["name"], job["items"], job["method"], job["m"], log,
                      cfg_cont=job.get("cfg_cont"), anchor=job.get("anchor", 0.0), lora=lora,
                      finalize=job.get("finalize", True))
        open(f"{Q}/{job['id']}.done", "w").write(a.worker)
    except Exception:
        tb = traceback.format_exc()
        print(tb, flush=True)
        open(f"{Q}/{job['id']}.failed", "w").write(tb)
        G.set_lora(None)
        torch.cuda.empty_cache()
open(f"{OUT}/logs/worker{a.worker}.done", "w").write("ok")
print(f"worker {a.worker} done", flush=True)

In [ ]:
%%writefile /kaggle/working/genesis5/code/genesis_metrics.py
"""genesis_metrics.py — quantitative evaluation for Genesis (IE 643): long videos made by chaining short clips.

Every stitched video comes with a manifest (clip lengths, clip boundaries, intentional [CUT] positions) and the
prompt of every clip. For each video we measure:

  Prompt adherence
    clipsim        CLIP-T: mean cosine(CLIP image feature of each frame, CLIP text feature of that clip's prompt) x 100
    prompt_top1    share of clips whose frames are closest (CLIP) to their OWN prompt among the video's prompts
    prompt_progress  mean over clips i>=1 of cos(clip_i, prompt_i) - cos(clip_i, prompt_0)  (x 100):
                   > 0 means the video moves on to the later prompts instead of staying on the first one
  Temporal consistency (VBench definitions, computed inside each shot)
    subject_cons   mean_t 0.5 [cos(d_t, d_0) + cos(d_t, d_{t-1})], d = DINO ViT-B/16 CLS feature
    background_cons  the same with CLIP image features
    start_end_id   cos(mean DINO feature of the first clip, mean DINO feature of the last clip)
  Transition smoothness
    seam_pix / seam_dino  mean frame-to-frame change AT the clip seams divided by the median change INSIDE clips
                   (pixel L1 / DINO 1-cos). 1.0 = a seam looks like any other frame step; a hard cut is >> 1.
                   Intentional [CUT]s are excluded (reported separately as cut_pix).
    seam_pct       percentile rank of the seam's pixel change among all within-clip frame changes
                   (~50 = an ordinary frame step, 100 = larger than every frame step inside the clips)
  Drift and video quality
    deltaE_final / deltaE_max  CIE-LAB distance between the mean colour of a clip and of the first clip of its shot
    overexp_final  % of pixels with L* > 95 in the last clip (blown-out highlights; Method A's typical failure)
    clipiqa / clipiqa_first / clipiqa_last  CLIP-IQA style zero-shot quality (antonym prompts, 0..1)
    sharpness      variance of the Laplacian (higher = more fine detail)
    motion         mean optical-flow magnitude (Farneback), pixels per frame at full resolution

Note: Method C pulls each frame's LAB mean/std toward the first clip, so deltaE is partly what C optimises.
overexp, clipiqa, start_end_id and the DINO/CLIP consistencies are independent of that correction.

Usage: python genesis_metrics.py --roots DIR [DIR ...] --out OUT_DIR [--device cuda]
Finds every logs/day*_plan.json under the roots; results are cached per video, so re-running only scores new videos.
"""
import argparse, glob, json, math, os, sys, time

import numpy as np
import torch
import torch.nn.functional as F

METHOD_ORDER = ["baseline", "A_lastframe", "B_m2", "B_m3", "C_anchor", "B_m2_lora", "C_anchor_lora"]
METHOD_COLOR = {"baseline": "#2a78d6", "A_lastframe": "#eb6834", "B_m2": "#1baf7a", "B_m3": "#eda100",
                "C_anchor": "#e87ba4", "B_m2_lora": "#008300", "C_anchor_lora": "#6250d6"}
METHOD_MARK = {"baseline": "o", "A_lastframe": "s", "B_m2": "^", "B_m3": "v", "C_anchor": "D",
               "B_m2_lora": "P", "C_anchor_lora": "X"}
OTHER_COLOR = "#8a8985"
IQA_PAIRS = [("Good photo.", "Bad photo."), ("Sharp photo.", "Blurry photo."),
             ("A photo with natural colors and exposure.", "An overexposed photo with washed-out colors.")]

# column -> (label, better direction) ; direction: +1 higher is better, -1 lower is better, 0 neither
COLUMNS = {
    "clipsim": ("CLIP-T x100", +1), "prompt_top1": ("Prompt top-1", +1), "prompt_progress": ("Prompt progress", +1),
    "subject_cons": ("Subject cons. (DINO)", +1), "background_cons": ("Background cons. (CLIP)", +1),
    "start_end_id": ("Start-end identity", +1), "seam_pix": ("Seam jump (pixel)", -1),
    "seam_dino": ("Seam jump (DINO)", -1), "seam_pct": ("Seam percentile", -1), "deltaE_final": ("Colour drift dE (last clip)", -1),
    "overexp_final": ("Overexposed % (last clip)", -1), "clipiqa": ("CLIP-IQA", +1),
    "clipiqa_drop": ("CLIP-IQA drop first->last", -1), "sharpness": ("Sharpness", +1),
    "motion": ("Motion (px/frame)", 0),
}


# ------------------------------------------------------------------------------------------------ models
class Models:
    """CLIP ViT-L/14 and DINO ViT-B/16 feature extractors (fp16 on GPU)."""

    def __init__(self, device="cuda", clip_id="openai/clip-vit-large-patch14", dino_id="facebook/dino-vitb16",
                 dtype=None):
        from transformers import AutoModel, CLIPModel, CLIPTokenizer
        self.device = device
        self.dtype = dtype or (torch.float16 if str(device).startswith("cuda") else torch.float32)
        def load(cls, mid):
            try:
                return cls.from_pretrained(mid, dtype=self.dtype)            # transformers >= 4.56
            except TypeError:
                return cls.from_pretrained(mid, torch_dtype=self.dtype)
        self.clip = load(CLIPModel, clip_id).to(device).eval()
        self.tok = CLIPTokenizer.from_pretrained(clip_id)
        self.dino = load(AutoModel, dino_id).to(device).eval()
        self.clip_size = self.clip.config.vision_config.image_size
        self.dino_size = getattr(self.dino.config, "image_size", 224)
        self.clip_mean, self.clip_std = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)
        self.dino_mean, self.dino_std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

    def _prep(self, frames_u8, size, mean, std):
        x = torch.from_numpy(np.ascontiguousarray(frames_u8)).to(self.device).permute(0, 3, 1, 2).float() / 255.0
        h, w = x.shape[-2:]
        s = size / min(h, w)
        x = F.interpolate(x, size=(max(size, round(h * s)), max(size, round(w * s))), mode="bicubic",
                          align_corners=False, antialias=True)
        h, w = x.shape[-2:]
        y0, x0 = (h - size) // 2, (w - size) // 2
        x = x[:, :, y0:y0 + size, x0:x0 + size]
        m = torch.tensor(mean, device=self.device).view(1, 3, 1, 1)
        sd = torch.tensor(std, device=self.device).view(1, 3, 1, 1)
        return ((x - m) / sd).to(self.dtype)

    @staticmethod
    def _tensor(out, *names):
        if torch.is_tensor(out):
            return out
        for n in names:
            v = getattr(out, n, None)
            if v is not None:
                return v
        raise TypeError(f"unexpected model output {type(out)}")

    @torch.no_grad()
    def image_clip(self, frames_u8, bs=32):
        outs = []
        for i in range(0, len(frames_u8), bs):
            px = self._prep(frames_u8[i:i + bs], self.clip_size, self.clip_mean, self.clip_std)
            f = self._tensor(self.clip.get_image_features(pixel_values=px), "image_embeds", "pooler_output")
            outs.append(F.normalize(f.float(), dim=-1).cpu())
        return torch.cat(outs)

    @torch.no_grad()
    def image_dino(self, frames_u8, bs=32):
        outs = []
        for i in range(0, len(frames_u8), bs):
            px = self._prep(frames_u8[i:i + bs], self.dino_size, self.dino_mean, self.dino_std)
            f = self.dino(pixel_values=px).last_hidden_state[:, 0]
            outs.append(F.normalize(f.float(), dim=-1).cpu())
        return torch.cat(outs)

    @torch.no_grad()
    def text_clip(self, texts):
        enc = self.tok(list(texts), padding=True, truncation=True, max_length=77, return_tensors="pt").to(self.device)
        f = self._tensor(self.clip.get_text_features(**enc), "text_embeds", "pooler_output")
        return F.normalize(f.float(), dim=-1).cpu()


# ------------------------------------------------------------------------------------------------ helpers
def read_u8(path):
    import imageio.v3 as iio
    return np.stack(list(iio.imiter(path)))


def _small(frames_u8, width=208):
    import cv2
    h, w = frames_u8.shape[1:3]
    size = (width, max(8, int(round(h * width / w))))
    return np.stack([cv2.resize(f, size, interpolation=cv2.INTER_AREA) for f in frames_u8])


def _lab(frames_small_u8):
    import cv2
    return np.stack([cv2.cvtColor(f.astype(np.float32) / 255.0, cv2.COLOR_RGB2LAB) for f in frames_small_u8])


def _vbench_consistency(feat):
    """VBench subject/background consistency for one shot: mean_t 0.5 [cos(f_t, f_0) + cos(f_t, f_{t-1})]."""
    if feat.shape[0] < 2:
        return float("nan")
    first = (feat[1:] @ feat[0]).clamp(min=0)
    prev = (feat[1:] * feat[:-1]).sum(-1).clamp(min=0)
    return float((0.5 * (first + prev)).mean())


def _segments(n_frames, boundaries):
    b = [x for x in boundaries if 0 < x < n_frames]
    starts = [0] + b
    ends = b + [n_frames]
    return list(zip(starts, ends))


def _nanmean(x):
    x = [v for v in x if v is not None and not (isinstance(v, float) and math.isnan(v))]
    return float(np.mean(x)) if x else float("nan")


# ------------------------------------------------------------------------------------------------ one video
def analyse_video(path, manifest, prompts, models, iqa_text=None):
    """Score one stitched video. Returns (video_record, [clip_records])."""
    import cv2
    t0 = time.time()
    fr = read_u8(path)
    N = fr.shape[0]
    fps = manifest.get("fps", 24)
    bounds = [int(b) for b in manifest.get("boundaries", []) if 0 < int(b) < N]
    cuts = set(int(c) for c in manifest.get("cuts", []))
    segs = _segments(N, bounds)
    n_clips = len(segs)
    prompts = list(prompts)[:n_clips] + [prompts[-1]] * max(0, n_clips - len(prompts)) if prompts else [""] * n_clips

    dino = models.image_dino(fr)
    clipf = models.image_clip(fr)
    small = _small(fr)
    lab = _lab(small)

    # ---- shots (split at intentional cuts) and per-clip shot reference
    shot_of_clip, shot = [], 0
    for i, (a, _) in enumerate(segs):
        if i > 0 and a in cuts:
            shot += 1
        shot_of_clip.append(shot)
    shots = {}
    for i, (a, b) in enumerate(segs):
        s = shot_of_clip[i]
        shots[s] = (shots[s][0], b) if s in shots else (a, b)

    # ---- temporal consistency (inside shots, frame-weighted)
    def per_shot(feat):
        vals, wts = [], []
        for a, b in shots.values():
            v = _vbench_consistency(feat[a:b])
            if not math.isnan(v):
                vals.append(v); wts.append(b - a)
        return float(np.average(vals, weights=wts)) if vals else float("nan")
    subject_cons, background_cons = per_shot(dino), per_shot(clipf)
    a0, b0 = segs[0]; aL, bL = segs[-1]
    start_end_id = float(F.normalize(dino[a0:b0].mean(0), dim=0) @ F.normalize(dino[aL:bL].mean(0), dim=0))

    # ---- transition smoothness
    sm = small.astype(np.float32) / 255.0
    d_pix = np.concatenate([[np.nan], np.abs(sm[1:] - sm[:-1]).mean(axis=(1, 2, 3))])
    d_dino = np.concatenate([[np.nan], (1 - (dino[1:] * dino[:-1]).sum(-1)).numpy()])
    inside = np.array([t for t in range(1, N) if t not in set(bounds)], dtype=int)
    trans = [b for b in bounds if b not in cuts]
    med_pix = float(np.median(d_pix[inside])) + 1e-6 if len(inside) else float("nan")
    med_dino = float(np.median(d_dino[inside])) + 1e-6 if len(inside) else float("nan")
    seam_pix = float(np.mean(d_pix[trans]) / med_pix) if trans else float("nan")
    seam_dino = float(np.mean(d_dino[trans]) / med_dino) if trans else float("nan")
    # percentile rank of each seam's pixel change among all within-clip changes (50 = ordinary step, 100 = biggest)
    seam_pct = (float(np.mean([(d_pix[inside] < d_pix[b]).mean() * 100 for b in trans]))
                if trans and len(inside) else float("nan"))
    cut_pix = float(np.mean(d_pix[sorted(cuts & set(bounds))]) / med_pix) if (cuts & set(bounds)) else float("nan")

    # ---- motion (optical flow on the small frames, pairs inside clips, every 2nd frame)
    grey = [cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in small]
    scale = fr.shape[2] / small.shape[2]
    bset = set(bounds)
    flow_mag = {}
    for t in range(1, N, 2):
        if t in bset:
            continue
        flow = cv2.calcOpticalFlowFarneback(grey[t - 1], grey[t], None, 0.5, 3, 15, 3, 5, 1.2, 0)
        flow_mag[t] = float(np.linalg.norm(flow, axis=-1).mean() * scale)
    motion = float(np.mean(list(flow_mag.values()))) if flow_mag else float("nan")

    # ---- prompt adherence
    T = models.text_clip(prompts)                                  # n_clips x D
    uniq = list(dict.fromkeys(prompts))
    TU = models.text_clip(uniq)
    clip_recs, top1, progress = [], [], []
    if iqa_text is None:
        iqa_text = models.text_clip([p for pair in IQA_PAIRS for p in pair])
    logit_scale = 100.0
    for i, (a, b) in enumerate(segs):
        f = clipf[a:b]
        cs = float((f @ T[i]).mean() * 100)
        mf = F.normalize(f.mean(0), dim=0)
        top1.append(float(uniq[int(torch.argmax(TU @ mf))] == prompts[i]))
        if i > 0 and prompts[i] != prompts[0]:
            progress.append(float((f @ T[i]).mean() - (f @ T[0]).mean()) * 100)
        # quality: CLIP-IQA style antonym prompts on every 4th frame
        sims = f[::4] @ iqa_text.T * logit_scale                     # k x (2*pairs)
        sims = sims.view(sims.shape[0], -1, 2)
        iqa = float(torch.softmax(sims, dim=-1)[..., 0].mean())
        # colour drift vs first clip of the same shot
        ref_clip = next(j for j in range(n_clips) if shot_of_clip[j] == shot_of_clip[i])
        ra, rb = segs[ref_clip]
        mu = lab[a:b].reshape(-1, 3).mean(0)
        mu_ref = lab[ra:rb].reshape(-1, 3).mean(0)
        L = lab[a:b, ..., 0]
        chroma = np.sqrt(lab[a:b, ..., 1] ** 2 + lab[a:b, ..., 2] ** 2)
        g_full = [cv2.cvtColor(x, cv2.COLOR_RGB2GRAY) for x in fr[a:b:8]]
        sharp = float(np.mean([cv2.Laplacian(g, cv2.CV_64F).var() for g in g_full]))
        mags_i = [v for t, v in flow_mag.items() if a < t < b]
        clip_recs.append({"clip": i, "t_start": round(a / fps, 2), "frames": int(b - a), "shot": shot_of_clip[i],
                          "prompt": prompts[i], "clipsim": cs, "clipiqa": iqa,
                          "deltaE": float(np.linalg.norm(mu - mu_ref)), "L": float(L.mean()),
                          "chroma": float(chroma.mean()), "overexp": float((L > 95).mean() * 100),
                          "underexp": float((L < 5).mean() * 100), "sharpness": sharp,
                          "motion": float(np.mean(mags_i)) if mags_i else float("nan")})

    rec = {"frames": int(N), "seconds": round(N / fps, 2), "n_clips": n_clips, "n_shots": len(shots),
           "clipsim": _nanmean([c["clipsim"] for c in clip_recs]),
           "prompt_top1": _nanmean(top1), "prompt_progress": _nanmean(progress),
           "subject_cons": subject_cons, "background_cons": background_cons, "start_end_id": start_end_id,
           "seam_pix": seam_pix, "seam_dino": seam_dino, "seam_pct": seam_pct, "cut_pix": cut_pix, "motion": motion,
           "deltaE_final": clip_recs[-1]["deltaE"], "deltaE_max": max(c["deltaE"] for c in clip_recs),
           "overexp_final": clip_recs[-1]["overexp"], "overexp_mean": _nanmean([c["overexp"] for c in clip_recs]),
           "clipiqa": _nanmean([c["clipiqa"] for c in clip_recs]), "clipiqa_first": clip_recs[0]["clipiqa"],
           "clipiqa_last": clip_recs[-1]["clipiqa"],
           "clipiqa_drop": clip_recs[0]["clipiqa"] - clip_recs[-1]["clipiqa"],
           "sharpness": _nanmean([c["sharpness"] for c in clip_recs]),
           "seam_frames": [int(b) for b in trans], "d_pix_at_seams": [float(d_pix[b]) for b in trans],
           "score_sec": round(time.time() - t0, 1)}
    return rec, clip_recs


# ------------------------------------------------------------------------------------------------ discovery
def discover(roots):
    """Yield dicts describing every stitched video found under the roots (one per manifest)."""
    seen_sets = {}
    for root in roots:
        for plan_p in sorted(glob.glob(os.path.join(root, "**", "logs", "day*_plan.json"), recursive=True)):
            run_dir = os.path.dirname(os.path.dirname(plan_p))
            set_name = os.path.basename(plan_p).replace("_plan.json", "")
            if set_name in seen_sets and seen_sets[set_name] != run_dir:
                print(f"[metrics] {set_name}: ignoring second copy at {run_dir} (using {seen_sets[set_name]})")
                continue
            seen_sets[set_name] = run_dir
            plan = json.load(open(plan_p))
            fps = plan.get("config", {}).get("FPS", 24)
            for man_p in sorted(glob.glob(os.path.join(run_dir, "manifests", "*.json"))):
                man = json.load(open(man_p))
                vid = os.path.join(run_dir, "videos", f"{man['seq']}_{man['method']}_FULL.mp4")
                if not os.path.exists(vid):
                    continue
                prompts = man.get("prompts") or plan.get("prompts", {}).get(man["seq"], [])
                man.setdefault("fps", fps)
                yield {"set": set_name, "seq": man["seq"], "method": man["method"], "video": vid,
                       "manifest": man, "prompts": prompts}


def score_all(roots, out, models=None, device="cuda", clip_id=None, dino_id=None):
    os.makedirs(f"{out}/cache", exist_ok=True)
    items = list(discover(roots))
    print(f"[metrics] {len(items)} videos found:",
          {s: sum(1 for x in items if x["set"] == s) for s in sorted({x["set"] for x in items})}, flush=True)
    todo = []
    for it in items:
        key = f"{it['set']}__{it['seq']}__{it['method']}"
        it["cache"] = f"{out}/cache/{key}.json"
        size = os.path.getsize(it["video"])
        if os.path.exists(it["cache"]) and json.load(open(it["cache"])).get("video_bytes") == size:
            continue
        todo.append(it)
    if todo:
        if models is None:
            kw = {k: v for k, v in (("clip_id", clip_id), ("dino_id", dino_id)) if v}
            models = Models(device=device, **kw)
        iqa_text = models.text_clip([p for pair in IQA_PAIRS for p in pair])
        for k, it in enumerate(todo):
            try:
                rec, clips = analyse_video(it["video"], it["manifest"], it["prompts"], models, iqa_text)
                meta = {"set": it["set"], "seq": it["seq"], "method": it["method"], "video": it["video"],
                        "video_bytes": os.path.getsize(it["video"]), "lora": it["manifest"].get("lora"),
                        "anchor": it["manifest"].get("anchor"), "m": it["manifest"].get("m")}
                json.dump({**meta, "video_metrics": rec, "clips": clips}, open(it["cache"], "w"), indent=1)
                print(f"[metrics] {k + 1}/{len(todo)} {it['set']} {it['seq']} {it['method']}: "
                      f"CLIP-T {rec['clipsim']:.1f} subj {rec['subject_cons']:.3f} seam {rec['seam_pix']:.2f} "
                      f"dE {rec['deltaE_final']:.1f} over {rec['overexp_final']:.1f}% ({rec['score_sec']}s)", flush=True)
            except Exception as e:
                print(f"[metrics] FAILED {it['video']}: {e!r}", flush=True)
    return build_report(out)


# ------------------------------------------------------------------------------------------------ report
def _order(methods):
    known = [m for m in METHOD_ORDER if m in methods]
    return known + sorted(m for m in methods if m not in METHOD_ORDER)


def build_report(out):
    import pandas as pd
    recs, clips = [], []
    for p in sorted(glob.glob(f"{out}/cache/*.json")):
        d = json.load(open(p))
        base = {k: d[k] for k in ("set", "seq", "method", "lora", "anchor", "m")}
        recs.append({**base, **{k: v for k, v in d["video_metrics"].items() if not isinstance(v, list)}})
        clips += [{**base, **c} for c in d["clips"]]
    if not recs:
        print("[metrics] nothing to report"); return None
    V, Cl = pd.DataFrame(recs), pd.DataFrame(clips)
    V.to_csv(f"{out}/per_video.csv", index=False)
    Cl.to_csv(f"{out}/per_clip.csv", index=False)
    cols = [c for c in COLUMNS if c in V.columns]
    md = ["# Genesis — quantitative results", "",
          "Mean over the prompt sequences of each set (± = standard deviation across sequences). "
          "Arrows: ↑ higher is better, ↓ lower is better. Seam jump: 1.0 = seam is as smooth as a normal frame step.", ""]
    for s in sorted(V["set"].unique()):
        Vs = V[V["set"] == s]
        g = Vs.groupby("method")[cols]
        mean, std, n = g.mean(), g.std(), Vs.groupby("method").size()
        mean = mean.loc[_order(list(mean.index))]
        mean.assign(n_videos=n).to_csv(f"{out}/summary_{s}.csv")
        md += [f"## {s}  ({', '.join(sorted(Vs['seq'].unique()))})", ""]
        hdr = ["method", "n"] + [COLUMNS[c][0] + {1: " ↑", -1: " ↓", 0: ""}[COLUMNS[c][1]] for c in cols]
        md += ["| " + " | ".join(hdr) + " |", "|" + "---|" * len(hdr)]
        for mth, row in mean.iterrows():
            cells = []
            for c in cols:
                v, sd = row[c], std.loc[mth, c] if mth in std.index else float("nan")
                f = "{:.3f}" if c in ("subject_cons", "background_cons", "start_end_id", "clipiqa", "clipiqa_drop",
                                      "prompt_top1") else ("{:.0f}" if c == "sharpness" else "{:.2f}")
                cells.append("–" if pd.isna(v) else (f.format(v) + ("" if pd.isna(sd) or n[mth] < 2 else " ± " + f.format(sd))))
            md.append("| " + " | ".join([mth, str(n[mth])] + cells) + " |")
        md.append("")
        try:
            make_figures(V, Cl, s, out)
        except Exception as e:
            print("[metrics] figure error", s, repr(e))
    open(f"{out}/summary.md", "w").write("\n".join(md))
    print("\n".join(md))
    return V, Cl


def _style(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_color("#b8b7b2")
    ax.tick_params(colors="#52514e", labelsize=8)
    ax.grid(axis="both", color="#ecebe7", linewidth=0.8)
    ax.set_axisbelow(True)


def make_figures(V, Cl, s, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
                         "axes.labelcolor": "#52514e", "text.color": "#0b0b0b", "figure.facecolor": "#fcfcfb",
                         "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb"})
    os.makedirs(f"{out}/figures", exist_ok=True)
    Vs, Cs = V[V["set"] == s], Cl[Cl["set"] == s]
    methods = _order(list(Vs["method"].unique()))
    col = lambda m: METHOD_COLOR.get(m, OTHER_COLOR)
    seqs = sorted(Vs["seq"].unique())

    # 1) metric panels: one small chart per metric, methods on the y axis, bar = mean, dots = sequences
    show = [c for c in ["clipsim", "subject_cons", "background_cons", "start_end_id", "seam_pct", "deltaE_final",
                        "overexp_final", "clipiqa", "motion", "prompt_top1"] if c in Vs.columns]
    ncol = 5
    nrow = math.ceil(len(show) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 0.32 * len(methods) * nrow + 1.2 * nrow + 0.4),
                             squeeze=False)
    for ax, c in zip(axes.flat, show):
        _style(ax); ax.grid(axis="y", visible=False)
        for y, m in enumerate(methods):
            vals = Vs[Vs["method"] == m][c].dropna()
            if len(vals) == 0:
                continue
            ax.scatter(vals, [y] * len(vals), s=14, color="#8a8985", alpha=0.8, zorder=2, linewidths=0)
            ax.scatter([vals.mean()], [y], s=90, color=col(m), marker=METHOD_MARK.get(m, "o"), zorder=3,
                       edgecolor="#fcfcfb", linewidth=2)
            ax.text(1.02, y, f"{vals.mean():.3g}", va="center", fontsize=7.5, color="#0b0b0b",
                    transform=ax.get_yaxis_transform())
        ax.set_yticks(range(len(methods)))
        ax.set_yticklabels(methods if ax in axes[:, 0] else [""] * len(methods))
        ax.invert_yaxis()
        lab, d = COLUMNS[c]
        ax.set_title(lab + {1: "  ↑", -1: "  ↓", 0: ""}[d], loc="left")
        if c == "seam_pct":
            ax.axvline(50.0, color="#52514e", linewidth=1, linestyle=":")
        ax.margins(x=0.12, y=0.12)
    for ax in list(axes.flat)[len(show):]:
        ax.axis("off")
    fig.suptitle(f"{s}: large marker = mean over sequences (value at right), grey dots = individual sequences", x=0.01, ha="left",
                 fontsize=10, color="#52514e")
    fig.tight_layout()
    fig.savefig(f"{out}/figures/metrics_{s}.png", dpi=160); plt.close(fig)

    # 2) drift over time: colour drift and overexposure per sequence (small multiples, one y-scale per chart)
    for c, lab in (("deltaE", "Colour drift ΔE vs first clip"), ("overexp", "Overexposed pixels (%)"),
                   ("clipiqa", "CLIP-IQA quality"), ("clipsim", "CLIP-T (prompt adherence) x100")):
        fig, axes = plt.subplots(1, len(seqs), figsize=(3.4 * len(seqs), 2.9), squeeze=False, sharey=True)
        for ax, q in zip(axes[0], seqs):
            _style(ax)
            for m in methods:
                d = Cs[(Cs["seq"] == q) & (Cs["method"] == m)].sort_values("clip")
                if len(d):
                    ax.plot(d["t_start"], d[c], color=col(m), linewidth=2, marker=METHOD_MARK.get(m, "o"),
                            markersize=5, markeredgecolor="#fcfcfb", markeredgewidth=1, label=m)
            ax.set_title(q, loc="left"); ax.set_xlabel("clip start (s)")
        axes[0][0].set_ylabel(lab)
        h, l = axes[0][0].get_legend_handles_labels()
        for ax in axes[0][1:]:
            hh, ll = ax.get_legend_handles_labels()
            for a_, b_ in zip(hh, ll):
                if b_ not in l:
                    h.append(a_); l.append(b_)
        fig.legend(h, l, loc="upper center", ncol=min(7, len(l)), frameon=False, fontsize=8,
                   bbox_to_anchor=(0.5, 1.0))
        fig.tight_layout(rect=(0, 0, 1, 0.88))
        fig.savefig(f"{out}/figures/{c}_over_time_{s}.png", dpi=160); plt.close(fig)

    # 3) trade-off: motion vs subject consistency (method means, directly labelled)
    g = Vs.groupby("method")[["motion", "subject_cons"]].mean()
    if len(g) >= 2 and g.notna().all().all():
        fig, ax = plt.subplots(figsize=(5.2, 3.8)); _style(ax)
        for m in methods:
            if m in g.index:
                ax.scatter(g.loc[m, "motion"], g.loc[m, "subject_cons"], s=70, color=col(m),
                           marker=METHOD_MARK.get(m, "o"), edgecolor="#fcfcfb", linewidth=2, zorder=3)
                ax.annotate(m, (g.loc[m, "motion"], g.loc[m, "subject_cons"]), xytext=(6, 4),
                            textcoords="offset points", fontsize=8, color="#0b0b0b")
        ax.set_xlabel("Motion (optical flow, px/frame) →  more dynamic")
        ax.set_ylabel("Subject consistency (DINO) →  more stable")
        ax.set_title(f"{s}: stability vs motion", loc="left")
        ax.margins(0.2)
        fig.tight_layout(); fig.savefig(f"{out}/figures/tradeoff_{s}.png", dpi=160); plt.close(fig)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--roots", nargs="+", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--clip", default=None)
    ap.add_argument("--dino", default=None)
    a = ap.parse_args()
    score_all(a.roots, a.out, device=a.device, clip_id=a.clip, dino_id=a.dino)

## 3 — Configuration: inputs from Day 2/3, prompts, jobs
`[CUT]` in a prompt list starts a new shot (a hard cut, generated from text). Everything between cuts is one
continuous take made with the chosen chaining method.

In [ ]:
import os, json, time, gc, shutil, re, html, sys, subprocess, glob, zipfile
import numpy as np
import torch

os.environ["HF_HOME"] = "/tmp/hf_home"
MODEL_ID, WEIGHTS_DIR = "Wan-AI/Wan2.2-TI2V-5B-Diffusers", "/tmp/wan22_ti2v5b"
OUT = "/kaggle/working/genesis5"
for d in ["embeds", "videos", "latents", "frames", "logs", "manifests", "code", "figures", "queue", "lora", "metrics"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)
sys.path.insert(0, f"{OUT}/code")
import genesis_lib as gl
N_GPU = torch.cuda.device_count()

RUN_TRAIN20 = False       # optional third showcase (new subject, +1.3 h); only if quota allows
GEN_HOURS = 2.75          # no new job starts unless it can finish within this many hours of launch
SEC_PER_CLIP = 420        # measured on Day 4 (~400 s per clip on a T4)
POLL = 300
METRIC_MODELS = []        # e.g. ["--clip", "openai/clip-vit-large-patch14", "--dino", "facebook/dino-vitb16"] (defaults)

HEIGHT, WIDTH, NUM_FRAMES, FPS = 480, 832, 49, 24
STEPS, GUIDANCE, FLOW_SHIFT, BASE_SEED = 30, 5.0, 3.0, 5000
CFG_CONT, ANCHOR, M = 3.5, 0.7, 2
NEG_PROMPT = ("Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, "
              "overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, "
              "poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, "
              "messy background, three legs, many people in the background, walking backwards")

# ---------------------------------------------------------------- inputs: earlier runs + the trained LoRA
LORA_OVERRIDE = None      # or a folder with lora.safetensors + lora_config.json (e.g. a Kaggle Dataset you uploaded)
INPUT_RUNS, lora_src = [], []
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    depth = root[len("/kaggle/input"):].count(os.sep)
    if depth <= 3:
        print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)" if files else "")
    if os.path.basename(root) == "logs" and any(re.fullmatch(r"day\d+_plan\.json", f) for f in files):
        INPUT_RUNS.append(os.path.dirname(root))
    if "lora.safetensors" in files and "lora_config.json" in files:
        lora_src.append(root)
INPUT_RUNS = sorted(set(r for r in INPUT_RUNS if not r.startswith(OUT)))
if LORA_OVERRIDE:
    lora_src = [LORA_OVERRIDE]
print("earlier runs found:", INPUT_RUNS)
print("LoRA folders found:", lora_src)
if not lora_src:
    raise SystemExit("STOP: the continuation LoRA was not found. Attach the Day 3 notebook under Input -> "
                     "Your Work -> Notebooks (or set LORA_OVERRIDE), then run again. Stopped early: no GPU time wasted.")
if not INPUT_RUNS:
    print("WARNING: no Day 2/3 videos attached -> only the new videos will be scored")
LORA_DIR = f"{OUT}/lora/genesis_continuation_lora"
if not os.path.exists(f"{LORA_DIR}/lora.safetensors"):
    shutil.copytree(lora_src[0], LORA_DIR, dirs_exist_ok=True)
HAVE_LORA = True
open(f"{OUT}/queue/lora.done", "w").write(lora_src[0])
print("continuation LoRA:", lora_src[0])

# ---------------------------------------------------------------- scripts
SIDE = "soft golden sunrise light, steady side tracking shot."
SCRIPTS = {
    "fox20b": [   # 20 s continuous take: 11 clips with m = 2; side view so the fox's shape stays readable
        "Side view of a red fox with a bushy white-tipped tail walking slowly from left to right through a snowy pine forest at sunrise, the whole fox clearly visible, soft golden light, steady side tracking shot.",
        f"Side view of the same red fox walking from left to right between the pine trees, its bushy tail held low behind it, {SIDE}",
        f"Side view of the same red fox slowing down to sniff the snow, then walking on from left to right, {SIDE}",
        f"Side view of the same red fox trotting a little faster from left to right through the snow, {SIDE}",
        f"Side view of the same red fox walking from left to right past a fallen log covered in snow, {SIDE}",
        f"Side view of the same red fox walking from left to right through deeper snow, leaving a trail of paw prints, {SIDE}",
        f"Side view of the same red fox stopping and lifting its head to listen, its ears pointed up, {SIDE}",
        f"Side view of the same red fox walking again from left to right up a gentle snowy slope, {SIDE}",
        f"Side view of the same red fox reaching the top of the slope as sunlight shines through the trees, {SIDE}",
        f"Side view of the same red fox standing still at the top of the snowy slope and looking around, {SIDE}",
        "Side view of the same red fox sitting down in the snow with its bushy tail curled around its paws, soft golden sunrise light, slow push-in shot.",
    ],
    "cutdemo": [   # 3 shots separated by [CUT]
        "Side view of a red fox walking slowly from left to right through a snowy pine forest at sunrise, soft golden light, wide shot.",
        "Side view of the same red fox walking from left to right through the snowy pine forest and stopping, soft golden light, wide shot.",
        "[CUT]",
        "Close-up of a red fox's face in a snowy forest at sunrise, its ears twitching as it listens, soft golden light, shallow depth of field.",
        "Close-up of the same red fox's face as it turns its head and sniffs the air, soft golden light, shallow depth of field.",
        "[CUT]",
        "A red fox runs away across a wide snowy clearing toward the pine forest at sunrise, soft golden light, wide shot from behind.",
        "The same red fox disappears between the pine trees at the edge of the snowy clearing, soft golden light, wide shot.",
    ],
    "train20": [
        "A red and black steam locomotive pulls a short train along a railway through green rolling hills on a sunny summer day, white steam rising, drone tracking shot from the side.",
        "The same red steam train keeps moving along the railway through the green hills, white steam drifting behind it, sunny day, drone tracking shot from the side.",
        "The same red steam train curves along the track beside a sparkling river, white steam rising, sunny day, drone tracking shot.",
        "The same red steam train approaches an old stone arch bridge over the river, sunny day, drone tracking shot.",
        "The same red steam train crosses the old stone arch bridge over the river, white steam trailing, sunny day, drone tracking shot.",
        "The same red steam train leaves the stone bridge and runs along the riverbank past green meadows, sunny day, drone tracking shot.",
        "The same red steam train passes a field of yellow flowers beside the track, white steam rising, sunny day, drone tracking shot.",
        "The same red steam train passes a small farmhouse with a red roof near the track, sunny day, drone tracking shot.",
        "The same red steam train climbs a gentle hill as the drone slowly rises higher, white steam rising, sunny day.",
        "The same red steam train approaches a small country station with a white platform, sunny day, drone tracking shot.",
        "The same red steam train slows down and stops at the small country station, steam drifting over the platform, sunny day, drone shot.",
    ],
}
ITEMS = {}
for j, (name, plist) in enumerate(SCRIPTS.items()):
    its = gl.parse_script(plist)
    for i, it in enumerate(its):
        it.update(key=f"{name}_{i}", seed=BASE_SEED + 100 * j + i)
    ITEMS[name] = its
n20 = gl.clips_for_duration(20, FPS, NUM_FRAMES, M)
assert len(ITEMS["fox20b"]) == n20, (len(ITEMS["fox20b"]), n20)
print(f"20 s at m={M} needs {n20} clips ->", (NUM_FRAMES + (n20 - 1) * gl.new_frames_per_clip(NUM_FRAMES, M)) / FPS, "s")

C_ARGS = dict(m=M, cfg_cont=CFG_CONT, anchor=ANCHOR)
def job(id_, name, method, deps, items=None, lora=False, finalize=True, chain=True):
    j = {"id": id_, "name": name, "items": items or ITEMS[name], "method": method, "lora": lora,
         "finalize": finalize, "deps": deps, "m": M}
    if chain:
        j.update(C_ARGS)
    return j
JOBS = [   # priority order: final method, [CUT] demo, then the no-LoRA ablation from the same first clip
    job("first_fox20b", "fox20b", "text", [], items=ITEMS["fox20b"][:1], finalize=False, chain=False),
    job("C_anchor_lora_fox20b", "fox20b", "C_anchor_lora", ["first_fox20b", "lora"], lora=True),
    job("C_anchor_lora_cutdemo", "cutdemo", "C_anchor_lora", ["lora"], lora=True),
    job("C_anchor_fox20b", "fox20b", "C_anchor", ["first_fox20b"]),
]
if RUN_TRAIN20:
    JOBS += [job("first_train20", "train20", "text", [], items=ITEMS["train20"][:1], finalize=False, chain=False),
             job("C_anchor_lora_train20", "train20", "C_anchor_lora", ["first_train20", "lora"], lora=True)]

CONFIG = {"OUT": OUT, "WEIGHTS_DIR": WEIGHTS_DIR, "EMBEDS": f"{OUT}/embeds/day5_embeds.pt", "HEIGHT": HEIGHT,
          "WIDTH": WIDTH, "NUM_FRAMES": NUM_FRAMES, "STEPS": STEPS, "GUIDANCE": GUIDANCE, "FLOW_SHIFT": FLOW_SHIFT,
          "FPS": FPS, "DEVICE": "cuda", "VAE_OFFLOAD": True, "DTYPE": "float16", "LORA_DIR": LORA_DIR,
          "SAVE_LATENTS": False, "DEADLINE": None, "SEC_PER_CLIP": SEC_PER_CLIP}
PLAN = f"{OUT}/logs/day5_plan.json"
json.dump({"config": CONFIG, "jobs": JOBS, "scripts": SCRIPTS, "negative_prompt": NEG_PROMPT}, open(PLAN, "w"), indent=2)
print(len(JOBS), "jobs:", [j["id"] for j in JOBS])

## 4 — Text encoder: encode every prompt, then free the memory

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, UMT5EncoderModel
import ftfy
snapshot_download(MODEL_ID, local_dir=WEIGHTS_DIR, allow_patterns=["model_index.json", "tokenizer/*", "text_encoder/*"])
tok = AutoTokenizer.from_pretrained(f"{WEIGHTS_DIR}/tokenizer")
if N_GPU >= 2:
    te = UMT5EncoderModel.from_pretrained(f"{WEIGHTS_DIR}/text_encoder", torch_dtype=torch.float32,
                                          device_map="auto", max_memory={0: "14GiB", 1: "14GiB", "cpu": "20GiB"})
    te_device = "cuda:0"
else:
    te = UMT5EncoderModel.from_pretrained(f"{WEIGHTS_DIR}/text_encoder", torch_dtype=torch.bfloat16); te_device = "cpu"
te.eval()

def prompt_clean(t):
    t = ftfy.fix_text(t); t = html.unescape(html.unescape(t)).strip()
    return re.sub(r"\s+", " ", t).strip()

@torch.no_grad()
def t5_embed(text, max_len=512):
    enc = tok([prompt_clean(text)], padding="max_length", max_length=max_len, truncation=True,
              add_special_tokens=True, return_attention_mask=True, return_tensors="pt")
    ids, mask = enc.input_ids.to(te_device), enc.attention_mask.to(te_device)
    n = int(mask.sum())
    h = te(ids, mask).last_hidden_state[0, :n].float().cpu()
    out = torch.zeros(1, max_len, h.shape[-1]); out[0, :n] = h
    return out.half()          # stored in fp16 (the transformer runs in fp16 anyway): half the file size

E = {"__negative__": t5_embed(NEG_PROMPT), "__empty__": t5_embed("")}
for name, its in ITEMS.items():
    for it in its:
        E[it["key"]] = t5_embed(it["prompt"])
torch.save(E, CONFIG["EMBEDS"]); print("prompts encoded:", len(E))
del te; gc.collect(); torch.cuda.empty_cache()
shutil.rmtree(f"{WEIGHTS_DIR}/text_encoder", ignore_errors=True); shutil.rmtree("/tmp/hf_home", ignore_errors=True)

## 5 — Download transformer, VAE and scheduler

In [ ]:
snapshot_download(MODEL_ID, local_dir=WEIGHTS_DIR, allow_patterns=["model_index.json", "transformer/*", "vae/*", "scheduler/*"])
!du -sh {WEIGHTS_DIR}/* | cat

## 6 — Run: GPU 0 generates right away; GPU 1 scores the Day 2/3 videos first, then generates
The metrics are zipped as soon as they finish (`genesis5_metrics.zip`), so they are safe even if generation is slow.

In [ ]:
env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false")
P = json.load(open(PLAN)); P["config"]["DEADLINE"] = time.time() + GEN_HOURS * 3600
json.dump(P, open(PLAN, "w"), indent=2)
for f in os.listdir(f"{OUT}/queue"):               # stale claims from an interrupted earlier run
    if f.endswith(".claimed") and not os.path.exists(f"{OUT}/queue/{f[:-8]}.done"):
        os.remove(f"{OUT}/queue/{f}")

def launch(args, gpu, tag):
    logf = open(f"{OUT}/logs/{tag}.log", "a")
    return subprocess.Popen([sys.executable, *args], env=dict(env, CUDA_VISIBLE_DEVICES=str(gpu)),
                            stdout=logf, stderr=subprocess.STDOUT)
def worker(w):
    return launch([f"{OUT}/code/genesis_queue_worker.py", "--plan", PLAN, "--worker", str(w)], w, f"worker{w}")
def metrics_cmd(roots):
    return [f"{OUT}/code/genesis_metrics.py", "--roots", *roots, "--out", f"{OUT}/metrics", *METRIC_MODELS]
def zip_dir(zpath, base, keep):
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(base):
            for fn in files:
                p = os.path.join(root, fn); rel = os.path.relpath(p, os.path.dirname(base))
                if keep(rel):
                    z.write(p, rel)
    print("wrote", zpath, f"{os.path.getsize(zpath) / 2**20:.1f} MB")
def last_line(tag):
    p = f"{OUT}/logs/{tag}.log"
    lines = open(p).read().strip().splitlines() if os.path.exists(p) else []
    return lines[-1][:140] if lines else "-"

procs = {"worker0": worker(0)}
w1 = N_GPU >= 2
if w1 and INPUT_RUNS:
    procs["metrics"] = launch(metrics_cmd(INPUT_RUNS), 1, "metrics")
elif w1:
    procs["worker1"] = worker(1)
t0 = time.time()
while True:
    m = procs.get("metrics")
    if w1 and "worker1" not in procs and m is not None and m.poll() is not None:
        print(open(f"{OUT}/logs/metrics.log").read()[-6000:], flush=True)
        zip_dir("/kaggle/working/genesis5_metrics.zip", f"{OUT}/metrics", lambda r: True)
        procs["worker1"] = worker(1)
    running = [k for k, p in procs.items() if p.poll() is None]
    if not running and (not w1 or "worker1" in procs):
        break
    time.sleep(POLL)
    print(f"[{(time.time() - t0) / 60:5.0f} min] " + " | ".join(f"{k}: {last_line(k)}" for k in procs), flush=True)
for k, p in procs.items():
    print(k, "exit code", p.returncode)
print({k: sorted(f[:-len(k)] for f in os.listdir(f'{OUT}/queue') if f.endswith(k)) for k in ['.done', '.failed', '.skipped']})

# score everything (earlier runs are cached, only the new videos are computed)
r = subprocess.run([sys.executable, *metrics_cmd(INPUT_RUNS + [OUT])], env=dict(env, CUDA_VISIBLE_DEVICES="0"),
                   capture_output=True, text=True)
open(f"{OUT}/logs/metrics_final.log", "w").write(r.stdout + r.stderr)
print((r.stdout + r.stderr)[-3000:])

## 7 — Results: metric tables and figures, showcase strips, comparison video, essentials zip

In [ ]:
from IPython.display import Markdown, Image as IPImage, display
from PIL import Image, ImageDraw
from diffusers.utils import export_to_video

if os.path.exists(f"{OUT}/metrics/summary.md"):
    display(Markdown(open(f"{OUT}/metrics/summary.md").read()))
for p in sorted(glob.glob(f"{OUT}/metrics/figures/metrics_*.png")) + sorted(glob.glob(f"{OUT}/metrics/figures/tradeoff_*.png")):
    print(os.path.basename(p)); display(IPImage(p, width=1100))

def label(img, text):
    im = Image.fromarray(img); d = ImageDraw.Draw(im)
    d.rectangle([0, 0, 8 + 7 * len(text), 18], fill=(0, 0, 0)); d.text((4, 3), text, fill=(255, 255, 255))
    return np.asarray(im)
def small(f, s=2):
    return np.asarray(Image.fromarray(gl.to_uint8(f)).resize((WIDTH // s, HEIGHT // s)))
def vid(name, method):
    p = f"{OUT}/videos/{name}_{method}_FULL.mp4"
    return gl.read_video(p) if os.path.exists(p) else None

# fox20b: final method vs no-LoRA ablation — timeline strip + side-by-side video
V = {k: vid("fox20b", k) for k in ["C_anchor_lora", "C_anchor"]}
V = {k: v for k, v in V.items() if v is not None}
if V:
    rows = []
    for k, v in V.items():
        ts = [min(int(s * FPS), v.shape[0] - 1) for s in (0, 4, 8, 12, 16, 20)]
        rows.append(np.concatenate([label(small(v[t], 3), f"{k} t={t / FPS:.0f}s") for t in ts], axis=1))
    Image.fromarray(np.concatenate(rows, 0)).save(f"{OUT}/figures/fox20b_timeline.jpg", quality=90)
    display(IPImage(f"{OUT}/figures/fox20b_timeline.jpg", width=1100))
    L = max(v.shape[0] for v in V.values())
    frames = [np.concatenate([label(small(v[min(t, v.shape[0] - 1)]), k) for k, v in V.items()], 1).astype(np.float32) / 255
              for t in range(L)]
    export_to_video(frames, f"{OUT}/videos/fox20b_COMPARE.mp4", fps=FPS)

# [CUT] demo and train20: frames around every seam / cut
for name in ["cutdemo", "train20"]:
    mp = f"{OUT}/manifests/{name}_C_anchor_lora.json"
    v = vid(name, "C_anchor_lora")
    if v is None or not os.path.exists(mp):
        continue
    man = json.load(open(mp)); rows = []
    for b in man["boundaries"]:
        kind = "CUT" if b in man.get("cuts", []) else "seam"
        rows.append(np.concatenate([label(small(v[j], 3), f"{kind}@{b} f{j}") for j in (b - 2, b - 1, b, b + 1)], 1))
    Image.fromarray(np.concatenate(rows, 0)).save(f"{OUT}/figures/{name}_seams.jpg", quality=88)
    print(name, f"{man['seconds']} s,", len(man["clip_lengths"]), "clips, cuts at", man.get("cuts", []))

recs = []
for f in glob.glob(f"{OUT}/logs/runs_worker*.jsonl"):
    recs += [json.loads(l) for l in open(f) if l.strip()]
new = [r for r in recs if "sec" in r]
if new:
    print(f"{len(new)} clips generated | mean {np.mean([r['sec'] for r in new]):.0f} s/clip | max peak "
          f"{max(r['peak_gb'] for r in new):.2f} GB | black/NaN: {[r['tag'] for r in new if not r.get('frames_ok', True)] or 'none'}")
json.dump(recs, open(f"{OUT}/logs/day5_runs_all.json", "w"), indent=1)
for f in os.listdir(f"{OUT}/queue"):
    if f.endswith(".failed") and f != "lora.failed":
        print("FAILED:", f, open(f"{OUT}/queue/{f}").read()[-1500:])

# keep the saved output small (large outputs made the Day 3 commit hang while saving)
for d in ["frames", "embeds", "latents", "lora"]:      # the LoRA itself stays in the Day 3 output
    shutil.rmtree(f"{OUT}/{d}", ignore_errors=True)
keep = lambda r: (r.endswith("_FULL.mp4") or r.endswith("_COMPARE.mp4") or not r.startswith("genesis5/videos/")) \
                 and not r.startswith("genesis5/lora/")
zip_dir("/kaggle/working/genesis5_essentials.zip", OUT, keep)
import psutil
for ch in psutil.Process().children(recursive=True):    # leftover helper processes can keep the kernel alive
    try:
        ch.kill()
    except Exception:
        pass
print("ALL DONE")
!du -sh {OUT} /kaggle/working/*.zip | cat

## What to send back
- `genesis5_essentials.zip` (Output tab): it has the metric tables (`metrics/summary.md`, CSVs), figures, the
  showcase videos and logs.
- If the zip is too big to upload, send `metrics/summary.md`, `metrics/figures/*.png`, `figures/fox20b_timeline.jpg`,
  `figures/cutdemo_seams.jpg` and the run-summary lines printed in step 7.

Next: the Gradio interface (prompts + duration + method + [CUT], live metrics) and the report.